# Gemma3-SD: Direct Gemma Conditioning for Stable Diffusion 1.5

**Path B/C hybrid — Direct runtime conditioning:** Replace CLIP cross-attention with Gemma 3 270M.
No runtime CLIP, no runtime adapter. CLIP is used once only for baked initialization, then deleted.

**Architecture:** Gemma 640-dim hidden states → UNet cross-attention `to_k`/`to_v` changed to 640 input dim.

**Training phases:**
- One-time baked CLIP→Gemma least-squares init (calibration only, no runtime adapter)
- Phase 1: Full-rank `to_k`/`to_v` warmup
- Phase 2: LoRA fine-tuning
- Save: research checkpoint + merged inference checkpoint + reload smoke-test cell


## Section 0: Google Drive Mount

Mount Google Drive for saving all artifacts (probe, LoRA, samples, complete model).

In [ ]:
# @title 0.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/gemma3-sd'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f"Artifacts will be saved to: {DRIVE_OUT}")


## Section 1: Environment Setup

In [ ]:
# @title 1.1 Simple install — Colab + Kohya, no Torch/NumPy changes

!pip install -q -U --upgrade-strategy only-if-needed \
  "Pillow==11.3.0" \
  "accelerate==1.6.0" \
  "transformers==4.54.1" \
  "diffusers[torch]==0.32.1" \
  "safetensors==0.4.5" \
  "datasets" \
  "peft" \
  "bitsandbytes" \
  "ftfy" \
  "einops" \
  "opencv-python==4.10.0.84" \
  "lion-pytorch" \
  "schedulefree" \
  "pytorch-optimizer" \
  "prodigyopt" \
  "prodigy-plus-schedule-free" \
  "toml" \
  "voluptuous" \
  "imagesize" \
  "rich" \
  "sentencepiece" \
  "wandb" \
  "matplotlib" \
  "tensorboard" \
  "tqdm"

!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts 2>/dev/null || true
!pip install -q --no-deps -e /content/sd-scripts

In [ ]:
import torch, numpy as np, PIL
import transformers, diffusers, accelerate

print("OK")
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

In [ ]:
# @title 1.3 HuggingFace Login via Colab Secrets
from google.colab import userdata
import os
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Missing HF_TOKEN in Colab secrets. Add it via the key icon in the sidebar.")
os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN: verified")
print("HF_TOKEN loaded from Colab secrets.")
print("Make sure you accepted the Gemma license: https://huggingface.co/google/gemma-3-270m-it")


In [ ]:
# @title 1.4 Setup wandb via Colab Secrets
import wandb
import os
from datetime import datetime

# Load wandb key from Colab secrets
wb_key = userdata.get("WANDB_API_KEY") or userdata.get("WANDB_KEY")
if wb_key is None:
    raise ValueError("Missing WANDB_API_KEY or WANDB_KEY in Colab secrets.")
os.environ["WANDB_API_KEY"] = wb_key
print("WANDB_API_KEY loaded from Colab secrets.")

run_name = f"gemma3-sd-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
wandb.init(
    project="gemma3-stable-diffusion",
    name=run_name,
    config={
        "model": "Gemma 3 270M → SD 1.5 (Path B)",
        "gemma_hidden_size": 640,
        "original_cross_attn_dim": 768,
        "new_cross_attn_dim": 640,
        "lora_rank": 8,
        "lora_alpha": 16,
        "learning_rate": 1e-4,
        "batch_size": 1,
        "max_prompt_length": 77,
        "device": str(torch.cuda.get_device_name(0)),
    }
)
print(f"wandb run: {wandb.run.name}")


In [ ]:
# @title 2.1 Load Gemma 3 270M
from transformers import AutoTokenizer, AutoModelForCausalLM
device = torch.device("cuda")
MAX_GEMMA_LEN = 77  # Match SD 1.5 conditioning length
# Gemma 3 270M — no CLIP loaded. UNet learns Gemma conditioning directly.
print("Loading Gemma 3 270M...")
gemma_path = "google/gemma-3-270m"
gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_path, token=os.environ.get("HF_TOKEN"))
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_path, torch_dtype=torch.bfloat16, device_map="auto",
        token=os.environ.get("HF_TOKEN"), low_cpu_mem_usage=True).eval()
gemma_hidden_size = gemma_model.config.hidden_size  # 640
print(f"  Gemma hidden_size: {gemma_hidden_size}")
print(f"  Gemma dtype: {next(gemma_model.parameters()).dtype}")
print("  No CLIP — SD UNet will learn Gemma language from scratch.")


## Section 2B: One-time CLIP→Gemma baked initialization (no runtime adapter)

Load CLIP ViT-L/14 temporarily, gather paired hidden states from a diverse
caption set, solve a least-squares mapping W: CLIP(768)→Gemma(640), then
bake W into the UNet cross-attention `to_k`/`to_v` weights and delete CLIP.
No adapter remains at inference time — the UNet learns Gemma directly.

In [ ]:
# @title 2.2 Load CLIP Temporarily (for calibration only — deleted after init)
import gc
from transformers import CLIPTextModel, CLIPTokenizer

print("Loading CLIP ViT-L/14 temporarily for calibration...")
clip_model = CLIPTextModel.from_pretrained(
    "openai/clip-vit-large-patch14", torch_dtype=torch.float16
).to(device).eval()
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
if clip_tokenizer.pad_token is None:
    clip_tokenizer.pad_token = clip_tokenizer.eos_token
print(f"  CLIP loaded. Hidden size: {clip_model.config.hidden_size}")
print(f"  Gemma hidden size: {gemma_hidden_size}")

In [ ]:
# @title 2.3 Gather Paired Hidden States (CLIP→Gemma calibration)
# Use 60+ diverse captions for representative hidden-state mapping.
# Split into train/validation so the W/b fit is not judged only in-sample.
CALIBRATION_CAPTIONS = [
    # --- General scenes ---
    "a beautiful sunset over the ocean with vibrant orange and pink clouds",
    "a serene mountain landscape with snow-capped peaks and a clear blue sky",
    "a dense forest with sunlight filtering through the canopy",
    "a bustling city street at night with neon signs and traffic",
    "a quiet countryside village with cobblestone streets and flower boxes",
    "an astronaut floating in space with Earth in the background",
    "a majestic castle on a hill surrounded by mist",
    "a tranquil lake reflecting the stars and a full moon",
    "a vibrant coral reef teeming with colorful fish",
    "a steampunk airship flying over a Victorian-era city",
    # --- Animals ---
    "a close-up portrait of a tiger with intense green eyes",
    "a playful kitten chasing a ball of yarn",
    "a majestic eagle soaring through a stormy sky",
    "a pack of wolves howling at the moon in a snowy forest",
    "a curious red fox peeking out from behind a tree",
    "a herd of elephants crossing a river at golden hour",
    "a hummingbird hovering near a bright tropical flower",
    "a bearded dragon basking on a warm rock",
    # --- People ---
    "a young woman reading a book in a cozy library",
    "an elderly man playing chess in a park",
    "a ballerina dancing gracefully on a dimly lit stage",
    "a samurai standing in a bamboo forest at dawn",
    "a cyberpunk hacker with neon-lit goggles in a dark room",
    "a medieval knight in shining armor holding a sword",
    "a chef cooking passionately in a rustic kitchen",
    "a painter working on a large canvas in a sunlit studio",
    # --- Objects & Food ---
    "a vintage typewriter on a wooden desk with scattered papers",
    "a steaming cup of coffee with latte art on a rainy morning",
    "a delicious plate of sushi arranged artistically on a black slate",
    "a classic red sports car parked on a coastal road",
    "a detailed pocket watch with exposed gears and Roman numerals",
    "a bouquet of wildflowers in a rustic clay vase",
    "a stack of old leather-bound books with gold lettering",
    # --- Fantasy/Sci-fi ---
    "a dragon breathing fire over a medieval village",
    "a futuristic city with flying cars and holographic billboards",
    "an enchanted forest with glowing mushrooms and fairy lights",
    "a space station orbiting a ringed gas giant planet",
    "a wizard casting a spell in a dark cavern filled with crystals",
    "a robot tending a garden in a post-apocalyptic world",
    "a phoenix rising from the ashes with fiery wings spread wide",
    "an underwater city with domed buildings and bioluminescent creatures",
    # --- Abstract/Artistic ---
    "an abstract painting with bold splashes of red, blue, and gold",
    "a minimalist black and white photograph of a spiral staircase",
    "a surreal dreamscape with floating islands and upside-down waterfalls",
    "a stained glass window depicting a celestial scene",
    "a 3D render of a geometric crystal formation in pastel colors",
    # --- Atmosphere/Mood ---
    "a rainy alleyway with puddles reflecting warm streetlights",
    "a cozy cabin interior with a crackling fireplace and snow outside",
    "a misty morning in a lavender field with soft purple hues",
    "a thunderstorm over a dark ocean with lightning striking the water",
    "a cherry blossom garden in full bloom with petals falling gently",
    "a desert landscape at noon with heat waves shimmering on the horizon",
    # --- Action/Dynamic ---
    "a surfer riding a massive wave at sunset",
    "a race car speeding through a tunnel with motion blur",
    "a waterfall cascading into a crystal-clear pool surrounded by moss",
    "a fireworks display illuminating a night sky over a city skyline",
    "a horse galloping across an open meadow with wind in its mane",
    # --- Validation-heavy diverse prompts ---
    "a microscopic view of a snowflake with intricate crystal patterns",
    "a macro photograph of a bee collecting pollen from a sunflower",
    "an ancient library with towering shelves and a spiral staircase",
    "a Japanese garden with a red bridge over a koi pond",
    "a glowing jellyfish floating in the deep dark ocean",
    "a bonsai tree on a stone pedestal in a zen garden",
]

CALIBRATION_VAL_COUNT = max(8, len(CALIBRATION_CAPTIONS) // 5)
train_captions = CALIBRATION_CAPTIONS[:-CALIBRATION_VAL_COUNT]
val_captions = CALIBRATION_CAPTIONS[-CALIBRATION_VAL_COUNT:]
print(f"Calibration captions: {len(CALIBRATION_CAPTIONS)} total = {len(train_captions)} train + {len(val_captions)} val")

@torch.no_grad()
def gather_pair(text):
    # CLIP
    clip_tokens = clip_tokenizer(
        text, return_tensors="pt", padding="max_length",
        max_length=77, truncation=True
    ).to(device)
    clip_out = clip_model(**clip_tokens)
    clip_hidden = clip_out.last_hidden_state  # [1, 77, 768]
    clip_mask = clip_tokens.attention_mask.bool()  # [1, 77]

    # Gemma
    gemma_tokens = gemma_tokenizer(
        text, return_tensors="pt", padding="max_length",
        max_length=MAX_GEMMA_LEN, truncation=True
    ).to(device)
    gemma_out = gemma_model(**gemma_tokens, output_hidden_states=True)
    gemma_hidden = gemma_out.hidden_states[-1]  # [1, N, 640]
    gemma_mask = gemma_tokens.attention_mask.bool()  # [1, N]

    # Align positions robustly. CLIP/Gemma tokenizers use different subwords,
    # so exact token semantics are rough; this is only a one-time linear warm-start.
    min_len = min(clip_hidden.shape[1], gemma_hidden.shape[1])
    clip_hidden = clip_hidden[:, :min_len, :]
    gemma_hidden = gemma_hidden[:, :min_len, :]
    clip_mask = clip_mask[:, :min_len]
    gemma_mask = gemma_mask[:, :min_len]

    valid = (clip_mask & gemma_mask)[0]
    if valid.sum() == 0:
        # Fallback for tokenizer/model versions whose attention masks are all-zero/odd.
        clip_ids = clip_tokens.input_ids[:, :min_len]
        gemma_ids = gemma_tokens.input_ids[:, :min_len]
        clip_pad = clip_tokenizer.pad_token_id
        gemma_pad = gemma_tokenizer.pad_token_id
        clip_count = int((clip_ids != clip_pad).sum().item()) if clip_pad is not None else min_len
        gemma_count = int((gemma_ids != gemma_pad).sum().item()) if gemma_pad is not None else min_len
        n_valid = min(min_len, clip_count, gemma_count)
        if n_valid <= 0:
            n_valid = min_len
        valid = torch.zeros(min_len, dtype=torch.bool, device=clip_hidden.device)
        valid[:n_valid] = True

    C_valid = clip_hidden[0, valid, :].float()   # [n_valid, 768]
    G_valid = gemma_hidden[0, valid, :].float()  # [n_valid, 640]
    return C_valid.cpu(), G_valid.cpu()

def gather_caption_split(captions, split_name):
    G_list, C_list = [], []
    skipped = 0
    for i, caption in enumerate(captions):
        C, G = gather_pair(caption)
        if C is not None and C.shape[0] > 0:
            G_list.append(G)
            C_list.append(C)
        else:
            skipped += 1
        if (i + 1) % 20 == 0 or (i + 1) == len(captions):
            print(f"  {split_name}: processed {i+1}/{len(captions)} captions...")
    if not G_list or not C_list:
        raise RuntimeError(
            f"Calibration split '{split_name}' produced zero paired hidden states. "
            "Check tokenizer attention masks and model outputs above."
        )
    G_all = torch.cat(G_list, dim=0)  # [total_valid, 640]
    C_all = torch.cat(C_list, dim=0)  # [total_valid, 768]
    print(f"  {split_name}: G={list(G_all.shape)}, C={list(C_all.shape)}, skipped={skipped}")
    return G_all, C_all, skipped

G_train, C_train, skipped_train = gather_caption_split(train_captions, "train")
G_val, C_val, skipped_val = gather_caption_split(val_captions, "val")

# Backward-compatible aliases used by the solve cell.
G_all, C_all = G_train, C_train
print(f"Train valid tokens: {G_train.shape[0]}; Val valid tokens: {G_val.shape[0]}")


In [ ]:
# @title 2.4 Solve W via Ridge Regression + Delete CLIP
RIDGE_LAMBDA = 1e-3

print("Solving W: C ≈ G @ M + b  ->  W = M.T")
# Train split only for fitting.
G = G_train.float()
C = C_train.float()

# Centered ridge: (G-G_mean) @ M ≈ (C-C_mean)
G_mean = G.mean(dim=0)  # [640]
C_mean = C.mean(dim=0)  # [768]
G_ctr = G - G_mean
C_ctr = C - C_mean
GTG = G_ctr.T @ G_ctr  # [640, 640]
GTG.diagonal().add_(RIDGE_LAMBDA)
GTC = G_ctr.T @ C_ctr  # [640, 768]

M = torch.linalg.solve(GTG, GTC)  # [640, 768]
W = M.T  # [768, 640]
b = C_mean - W @ G_mean  # [768] (CLIP-space bias)

def calibration_metrics(G_eval, C_eval, split_name):
    G_eval = G_eval.float()
    C_eval = C_eval.float()
    C_pred = G_eval @ M + b.unsqueeze(0)  # [N, 768]
    mse = torch.mean((C_eval - C_pred) ** 2)
    ss_res = ((C_eval - C_pred) ** 2).sum()
    ss_tot = ((C_eval - C_eval.mean(dim=0, keepdim=True)) ** 2).sum().clamp_min(1e-8)
    r2 = 1 - ss_res / ss_tot
    cosine = torch.nn.functional.cosine_similarity(C_pred, C_eval, dim=-1).mean()
    print(f"  {split_name}: R²={r2.item():.4f}, MSE={mse.item():.6f}, cosine={cosine.item():.4f}")
    return {"r2": float(r2.item()), "mse": float(mse.item()), "cosine": float(cosine.item())}

train_metrics = calibration_metrics(G_train, C_train, "train")
val_metrics = calibration_metrics(G_val, C_val, "val")

print(f"  W shape: {list(W.shape)}  (CLIP 768 ← Gemma 640)")
print(f"  b shape: {list(b.shape)}  (CLIP 768 bias)")
if val_metrics["cosine"] < 0.20:
    print("  WARNING: low validation cosine; baked init may be weak. Training may need more steps.")

# Save calibration artifact with enough metadata to reproduce/evaluate the bake.
import os, hashlib, json as _json
captions_hash = hashlib.sha256(_json.dumps(CALIBRATION_CAPTIONS, sort_keys=True).encode()).hexdigest()
save_path = f"{DRIVE_OUT}/clip_gemma_calib.pt"
torch.save({
    "W": W.cpu(),
    "b": b.cpu(),
    "M": M.cpu(),
    "ridge_lambda": RIDGE_LAMBDA,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "captions_hash": captions_hash,
    "num_train_captions": len(train_captions),
    "num_val_captions": len(val_captions),
    "gemma_model_id": gemma_path,
    "clip_model_id": "openai/clip-vit-large-patch14",
    "gemma_hidden_size": gemma_hidden_size,
    "clip_hidden_size": clip_hidden_size,
    "max_gemma_len": MAX_GEMMA_LEN,
}, save_path)
print(f"  Saved calibration to: {save_path}")

# Delete CLIP and intermediate tensors. Runtime training/inference use Gemma only.
del clip_model, clip_tokenizer, G_train, C_train, G_val, C_val, G_all, C_all, G, C, G_ctr, C_ctr, GTG, GTC, M
try:
    del C_pred
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("  CLIP deleted. Memory freed.")


## Section 3: UNet Surgery + Gemma Norm

Replace every cross-attention `to_k`/`to_v` from 768-dim CLIP input to 640-dim Gemma input.
New layers use baked CLIP→Gemma least-squares init — one-time W/b projection, no runtime adapter.
The replacement K/V layers intentionally keep bias=True so the calibration intercept is preserved.


In [ ]:
# @title 3.1 Load SD 1.5 UNet + VAE (no CLIP, no pipeline)
# Load only the components we need — skip CLIP text encoder, tokenizer, safety checker.
# This saves ~2 GB RAM vs loading the full StableDiffusionPipeline.
from diffusers import UNet2DConditionModel, AutoencoderKL

SD_ID = "runwayml/stable-diffusion-v1-5"

unet = UNet2DConditionModel.from_pretrained(
    SD_ID, subfolder="unet", torch_dtype=torch.float16
).to(device)

vae = AutoencoderKL.from_pretrained(
    SD_ID, subfolder="vae", torch_dtype=torch.float16
).to(device)

unet_dtype = next(unet.parameters()).dtype
print(f"  UNet cross_attention_dim: {unet.config.cross_attention_dim}")  # 768
print(f"  UNet dtype: {unet_dtype}")
print(f"  VAE dtype: {next(vae.parameters()).dtype}")
print("  No runtime CLIP loaded here — UNet will use Gemma conditioning. CLIP was calibration-only.")


In [ ]:
# @title 3.2 UNet Surgery: Replace Cross-Attention + Gemma Scale (baked CLIP\u2192Gemma init)
import torch.nn as nn

new_dim = gemma_hidden_size  # 640 (Gemma 3 270M)
unet_dtype = next(unet.parameters()).dtype
print(f"  UNet dtype: {unet_dtype}")
old_dim = unet.config.cross_attention_dim  # 768 (original CLIP)
print(f"Replacing cross-attn: Linear({old_dim}, *) -> Linear({new_dim}, *)")

# Fixed scalar (non-learnable) to stabilize Gemma activation scale
gemma_scale = 1.0  # Fixed scalar norm

# Load baked CLIP→Gemma calibration
calib_path = f"{DRIVE_OUT}/clip_gemma_calib.pt"
used_kaiming_fallback = False

if os.path.exists(calib_path):
    calib = torch.load(calib_path, map_location="cpu")
    W = calib["W"].to(device=device, dtype=unet_dtype)  # [768, 640]
    b = calib["b"].to(device=device, dtype=unet_dtype)  # [768]
    print(f"  Loaded baked init: W {list(W.shape)}, b {list(b.shape)}")
elif 'W' in globals() and 'b' in globals():
    W = globals()['W']
    b = globals()['b']
    # Ensure correct device/dtype
    if W.device != device or W.dtype != unet_dtype:
        W = W.to(device=device, dtype=unet_dtype)
    if b.device != device or b.dtype != unet_dtype:
        b = b.to(device=device, dtype=unet_dtype)
    print("  Using in-memory W/b from calibration step.")
else:
    W = None
    b = None
    print("  *** WARNING: No W available — falling back to Kaiming init! ***")
    used_kaiming_fallback = True

replacements = 0
for name, module in unet.named_modules():
    if hasattr(module, "attn2") and module.attn2 is not None:
        attn = module.attn2
        inner_dim = attn.to_k.out_features

        if not used_kaiming_fallback and W is not None:
            # --- Baked CLIP\u2192Gemma init ---
            # Clone old weights before replacement
            old_k_weight = attn.to_k.weight.data.clone()  # [inner_dim, 768]
            old_v_weight = attn.to_v.weight.data.clone()  # [inner_dim, 768]
            old_k_bias = attn.to_k.bias.data.clone() if attn.to_k.bias is not None else None
            old_v_bias = attn.to_v.bias.data.clone() if attn.to_v.bias is not None else None

            # new_weight = old_weight @ W
            new_k_weight = old_k_weight @ W  # [inner_dim, 768] @ [768, 640] = [inner_dim, 640]
            new_v_weight = old_v_weight @ W

            # to_k
            # Force bias=True so the CLIP-space intercept b is not silently dropped.
            k_bias_flag = True
            new_k = nn.Linear(new_dim, inner_dim, bias=k_bias_flag, device=device, dtype=unet_dtype)
            new_k.weight.data.copy_(new_k_weight.to(device=device, dtype=unet_dtype))
            # new_bias = old_w @ b + old_bias (old SD1.5 bias is usually None/zero)
            old_k_bias_term = old_k_bias if old_k_bias is not None else torch.zeros(inner_dim, device=old_k_weight.device, dtype=old_k_weight.dtype)
            new_k_bias = old_k_weight @ b + old_k_bias_term
            new_k.bias.data.copy_(new_k_bias.to(device=device, dtype=unet_dtype))
            attn.to_k = new_k

            # to_v
            # Force bias=True so the CLIP-space intercept b is not silently dropped.
            v_bias_flag = True
            new_v = nn.Linear(new_dim, inner_dim, bias=v_bias_flag, device=device, dtype=unet_dtype)
            new_v.weight.data.copy_(new_v_weight.to(device=device, dtype=unet_dtype))
            old_v_bias_term = old_v_bias if old_v_bias is not None else torch.zeros(inner_dim, device=old_v_weight.device, dtype=old_v_weight.dtype)
            new_v_bias = old_v_weight @ b + old_v_bias_term
            new_v.bias.data.copy_(new_v_bias.to(device=device, dtype=unet_dtype))
            attn.to_v = new_v
        else:
            # --- Kaiming fallback ---
            bias = attn.to_k.bias is not None
            new_k = nn.Linear(new_dim, inner_dim, bias=bias, device=device, dtype=unet_dtype)
            nn.init.kaiming_uniform_(new_k.weight, a=5**0.5)
            attn.to_k = new_k

            bias = attn.to_v.bias is not None
            new_v = nn.Linear(new_dim, inner_dim, bias=bias, device=device, dtype=unet_dtype)
            nn.init.kaiming_uniform_(new_v.weight, a=5**0.5)
            attn.to_v = new_v

        replacements += 1

print(f"  Replaced {replacements} to_k/to_v pairs")
if used_kaiming_fallback:
    print("  Init: Kaiming uniform (FALLBACK \u2014 calibration missing)")
else:
    print("  Init: Baked CLIP\u2192Gemma least-squares projection")
print(f"  Gemma scale: fixed scalar = {gemma_scale}")

# Verify no NaN
for n, p in unet.named_parameters():
    if "to_k" in n or "to_v" in n:
        assert p.isfinite().all(), f"{n} has NaN/Inf"
print("  All new to_k/to_v weights finite")
unet.register_to_config(cross_attention_dim=new_dim)
print(f"  Updated unet.config.cross_attention_dim: {unet.config.cross_attention_dim}")


In [ ]:
# @title 3.3 Verify Forward Pass
@torch.no_grad()
def test_forward(prompt="a cat on a table"):
    tokens = gemma_tokenizer(prompt, return_tensors="pt", padding="max_length",
                            truncation=True, max_length=MAX_GEMMA_LEN).to(device)
    out = gemma_model(**tokens, output_hidden_states=True)
    ehs = out.hidden_states[-1].to(dtype=unet_dtype)
    ehs = ehs * gemma_scale

    latents = torch.randn(1, 4, 64, 64, device=device, dtype=unet_dtype)
    t = torch.tensor([500], device=device)
    result = unet(latents, t, encoder_hidden_states=ehs,
                  encoder_attention_mask=tokens.attention_mask).sample
    print(f"Input: {ehs.shape}, Output: {result.shape}")
    return result

test_forward()
print("\u2713 Forward pass OK!")


## Section 4: LoRA Training

Freeze VAE + all UNet except LoRA on attn2.to_k/attn2.to_v

## Section 4A: Streaming Dataset

Streams `jackyhate/text-to-image-2M` with simple custom aspect-ratio bucketing.
This is **not** sd-scripts `BucketManager`; it is a minimal custom loop for proof-of-life training.
With `batch_size=1`, each sample may use its own bucket resolution without padding.


In [ ]:
# @title 4.2 Streaming IterableDataset
import io
import torch
from PIL import Image
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms
from datasets import load_dataset

BUCKETS = [
    (512, 512), (512, 768), (768, 512),
    (448, 704), (704, 448), (384, 640), (640, 384),
]

def get_bucket(w, h):
    """Nearest aspect-ratio bucket."""
    target = w / h
    best, best_dist = None, float("inf")
    for bw, bh in BUCKETS:
        dist = abs(bw / bh - target)
        if dist < best_dist:
            best_dist, best = dist, (bw, bh)
    return best

class StreamingSDDataset(IterableDataset):
    """Stream images from HF webdataset. No disk cache."""
    def __init__(self, ds_iter, tokenizer, vae=None, max_samples=2000, max_length=77):
        self.ds_iter = ds_iter
        self.max_samples = max_samples
        self.tokenizer = tokenizer
        self.vae = vae
        self.max_length = max_length

    def __iter__(self):
        import json as _json

        def _get_caption(sample):
            meta = sample.get("json", {})
            if isinstance(meta, bytes):
                meta = meta.decode("utf-8", errors="ignore")
            if isinstance(meta, str):
                try:
                    meta = _json.loads(meta)
                except Exception:
                    return ""
            if isinstance(meta, dict):
                return meta.get("prompt") or meta.get("caption") or meta.get("text") or ""
            return ""

        def _get_image(sample):
            for key in ["jpg", "jpeg", "png", "webp", "image"]:
                img = sample.get(key)
                if img is not None:
                    return img
            return None

        worker_info = torch.utils.data.get_worker_info()
        per_worker = self.max_samples
        if worker_info is not None:
            per_worker = self.max_samples // worker_info.num_workers

        count = 0
        for sample in self.ds_iter:
            if count >= per_worker:
                break
            caption = _get_caption(sample)
            if not caption:
                continue
            img = _get_image(sample)
            if img is None:
                continue
            if isinstance(img, bytes):
                img = Image.open(io.BytesIO(img))
            img = img.convert("RGB")

            bw, bh = get_bucket(img.width, img.height)
            b_ratio = bw / bh
            w, h = img.size
            if w / h > b_ratio:
                new_w = int(h * b_ratio)
                img = img.crop(((w - new_w) // 2, 0, (w + new_w) // 2, h))
            else:
                new_h = int(w / b_ratio)
                img = img.crop((0, (h - new_h) // 2, w, (h + new_h) // 2))
            img = img.resize((bw, bh), Image.LANCZOS)

            tokens = self.tokenizer(caption, return_tensors="pt",
                                   padding="max_length", max_length=self.max_length,
                                   truncation=True)
            img_tensor = transforms.ToTensor()(img) * 2 - 1
            yield {
                "image": img_tensor,
                "input_ids": tokens.input_ids[0],
                "attention_mask": tokens.attention_mask[0],
            }
            count += 1

MAX_SAMPLES = 2_000
STREAM_REPO = "jackyhate/text-to-image-2M"
print(f"Streaming from: {STREAM_REPO}")
ds_full = load_dataset(STREAM_REPO, streaming=True, split="train")
it = iter(ds_full)
s0 = next(it)
json_data = s0.get("json", {})
prompt = json_data.get("prompt", "") if isinstance(json_data, dict) else str(json_data)
print(f"  Sample keys: {list(s0.keys())}")
print(f"  Prompt: {prompt[:80]}")
ds = StreamingSDDataset(ds_full, gemma_tokenizer, max_samples=MAX_SAMPLES, max_length=MAX_GEMMA_LEN)
dl = DataLoader(ds, batch_size=1, num_workers=0)
print(f"Dataset ready: {MAX_SAMPLES} samples, {len(BUCKETS)} buckets")



## Section 4: Phase 1 — Full-Rank Cross-Attn Training

Train the replaced `attn2.to_k`/`attn2.to_v` layers full-rank first.
LoRA is too restrictive for freshly initialized cross-attention weights.
After warmup, freeze and switch to LoRA for fine-tuning.


In [ ]:
# @title 4.0 Full-Rank Cross-Attn Warmup (quick test: 1 epoch)
from diffusers import DDPMScheduler
from tqdm import tqdm

FULLRANK_EPOCHS = 1
FULLRANK_LR = 1e-4

# Freeze all UNet except attn2.to_k/to_v
for p in unet.parameters():
    p.requires_grad = False

fullrank_params = []
for name, module in unet.named_modules():
    if "attn2" in name and hasattr(module, "to_k"):
        for p in module.to_k.parameters():
            p.requires_grad = True
            fullrank_params.append(p)
        for p in module.to_v.parameters():
            p.requires_grad = True
            fullrank_params.append(p)

print(f"Full-rank trainable params: {sum(p.numel() for p in fullrank_params):,}")

scheduler = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")
optimizer = torch.optim.AdamW(fullrank_params, lr=FULLRANK_LR)
unet.train()
vae.eval()

print(f"Full-rank warmup: {FULLRANK_EPOCHS} epochs")
for epoch in range(FULLRANK_EPOCHS):
    # Refresh streaming iterator each epoch — HF iterators exhaust
    ds_full = load_dataset("jackyhate/text-to-image-2M", split="train", streaming=True)
    ds = StreamingSDDataset(ds_full, tokenizer=gemma_tokenizer,
                           vae=vae, max_samples=MAX_SAMPLES,
                           max_length=MAX_GEMMA_LEN)
    dl = DataLoader(ds, batch_size=1, num_workers=0)
    total_loss = 0.0
    seen = 0
    progress = tqdm(dl, desc=f"Warmup {epoch+1}/{FULLRANK_EPOCHS}")
    for batch in progress:
        img = batch["image"].to(device, dtype=torch.float16)
        with torch.no_grad():
            latent = vae.encode(img).latent_dist.sample() * vae.config.scaling_factor

            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)

            if torch.rand(()).item() < 0.10:
                empty = gemma_tokenizer([""], return_tensors="pt", padding="max_length",
                                        truncation=True, max_length=MAX_GEMMA_LEN).to(device)
                input_ids = empty.input_ids.expand(input_ids.shape[0], -1)
                attn_mask = empty.attention_mask.expand(attn_mask.shape[0], -1)

        with torch.no_grad():
            gemma_out = gemma_model(input_ids=input_ids, attention_mask=attn_mask, output_hidden_states=True)
            ehs = gemma_out.hidden_states[-1].to(dtype=unet_dtype)
            ehs = ehs * gemma_scale

        noise = torch.randn_like(latent)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (latent.shape[0],), device=device).long()
        noisy = scheduler.add_noise(latent, noise, t)
        pred = unet(noisy, t, encoder_hidden_states=ehs, encoder_attention_mask=attn_mask).sample
        loss = nn.functional.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(fullrank_params, 1.0)
        optimizer.step()

        total_loss += loss.item()
        seen += 1
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    print(f"Warmup epoch {epoch+1}: avg_loss = {total_loss/max(seen,1):.4f}")
    wandb.log({"warmup/loss": total_loss/max(seen,1), "warmup/epoch": epoch})

print("Full-rank warmup complete. Switching to LoRA...")


In [ ]:
# @title 4.1 Manual LoRA for Cross-Attention Only
class ManualLoRA(nn.Module):
    """LoRA wrapper for a single Linear layer."""
    def __init__(self, base_linear, rank=8, alpha=16):
        super().__init__()
        self.base = base_linear
        self.rank = rank
        self.scaling = alpha / rank
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_B.weight)
        self.lora_A = self.lora_A.to(dtype=base_linear.weight.dtype, device=base_linear.weight.device)
        self.lora_B = self.lora_B.to(dtype=base_linear.weight.dtype, device=base_linear.weight.device)
        for p in base_linear.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(x)) * self.scaling

    def get_peft_state_dict(self):
        """Export LoRA weights in peft-compatible format."""
        return {
            "lora_A.weight": self.lora_A.weight.data.clone(),
            "lora_B.weight": self.lora_B.weight.data.clone(),
            "r": self.rank,
            "lora_alpha": int(self.scaling * self.rank),
        }

# Freeze everything first
for p in unet.parameters():
    p.requires_grad = False

# Apply LoRA only to attn2 to_k and to_v
lora_count = 0
for name, module in unet.named_modules():
    if hasattr(module, 'to_k') and 'attn2' in name and module.to_k.in_features == new_dim:
        module.to_k = ManualLoRA(module.to_k, rank=8, alpha=16)
        module.to_v = ManualLoRA(module.to_v, rank=8, alpha=16)
        lora_count += 2

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f"LoRA layers: {lora_count}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
wandb.log({"lora_layers": lora_count, "trainable_params": trainable, "total_params": total})

## Section 4B: Phase 2 — LoRA Training

VAE encoding happens on-the-fly in the training loop (below).
No precomputation — each sample is encoded fresh, avoiding frozen noise.
Cost: ~3ms per sample on T4 (negligible vs Gemma's 20-50ms forward pass).


In [ ]:
# @title 4.4 Training Loop
from diffusers import DDPMScheduler
from tqdm import tqdm

scheduler = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")
optimizer = torch.optim.AdamW([p for p in unet.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)

unet.train()
vae.eval()
num_epochs = 1
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch_size = 4
global_step = 0

print(f"Training {num_epochs} epochs, streaming from webdataset")
print(f"  Grad accum: {GRADIENT_ACCUMULATION_STEPS}x, effective batch = {GRADIENT_ACCUMULATION_STEPS}")
# wandb.watch disabled — expensive on large UNet

try:
    for epoch in range(num_epochs):
        # Refresh streaming iterator each epoch — HF iterators exhaust
        ds_full = load_dataset("jackyhate/text-to-image-2M", split="train", streaming=True)
        ds = StreamingSDDataset(ds_full, tokenizer=gemma_tokenizer,
                               vae=vae, max_samples=MAX_SAMPLES,
                               max_length=MAX_GEMMA_LEN)
        dl = DataLoader(ds, batch_size=1, num_workers=0)
        epoch_loss = 0.0
        samples_seen = 0
        progress = tqdm(dl, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch in progress:
            # VAE encode on-the-fly
            img = batch["image"].to(device, dtype=torch.float16)
            with torch.no_grad():
                latent_dist = vae.encode(img).latent_dist
                latent = latent_dist.sample() * vae.config.scaling_factor

            # Gemma text encoding
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)

            # CFG dropout: 10% chance of empty prompt for unconditional training
            if torch.rand(()).item() < 0.10:
                empty = gemma_tokenizer([""], return_tensors="pt", padding="max_length",
                                        truncation=True, max_length=MAX_GEMMA_LEN).to(device)
                input_ids = empty.input_ids.expand(input_ids.shape[0], -1)
                attn_mask = empty.attention_mask.expand(attn_mask.shape[0], -1)

            with torch.no_grad():
                gemma_out = gemma_model(input_ids=input_ids, attention_mask=attn_mask, output_hidden_states=True)
                ehs = gemma_out.hidden_states[-1].to(dtype=unet_dtype)
                ehs = ehs * gemma_scale

            # Noise + UNet forward
            noise = torch.randn_like(latent)
            t = torch.randint(0, scheduler.config.num_train_timesteps, (latent.shape[0],), device=device).long()
            noisy = scheduler.add_noise(latent, noise, t)
            pred = unet(noisy, t, encoder_hidden_states=ehs, encoder_attention_mask=attn_mask).sample
            loss = nn.functional.mse_loss(pred.float(), noise.float())

            # Gradient accumulation
            loss = loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()

            if (global_step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                nn.utils.clip_grad_norm_([p for p in unet.parameters() if p.requires_grad], 1.0)
                optimizer.step()
                optimizer.zero_grad()

            epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
            samples_seen += 1
            global_step += 1
            progress.set_postfix({"loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})

            if global_step % 10 == 0:
                wandb.log({"train/loss": loss.item() * GRADIENT_ACCUMULATION_STEPS, "train/step": global_step})

        avg_loss = epoch_loss / max(samples_seen, 1)
        print(f"Epoch {epoch+1}: avg_loss = {avg_loss:.4f}")
        wandb.log({"train/epoch_loss": avg_loss, "train/epoch": epoch})

        if (epoch + 1) % 5 == 0:
            lora_w = {}
            for n, p in unet.named_parameters():
                if "lora_A" in n or "lora_B" in n:
                    lora_w[n] = p.data.cpu().clone()
            torch.save(lora_w, f"{DRIVE_OUT}/lora_epoch{epoch+1}.pt")
            print("  Checkpoint saved")

    # Save final LoRA
    lora_w = {}
    for n, p in unet.named_parameters():
        if "lora_A" in n or "lora_B" in n:
            lora_w[n] = p.data.cpu().clone()
    torch.save(lora_w, f"{DRIVE_OUT}/gemma3_sd_lora.pt")
    print("Training complete! LoRA saved.")
finally:
    wandb.finish()


## Section 5: Inference

Generate images with the Gemma-conditioned SD.

In [ ]:
# @title 5.1 Generate Image
import matplotlib.pyplot as plt
from diffusers import DPMSolverMultistepScheduler
from tqdm import tqdm

vae.to(device).eval()
unet.eval()
scheduler = DPMSolverMultistepScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")

@torch.no_grad()
def generate(prompt, steps=30, guidance=7.5, seed=42):
    gen = torch.Generator(device=device).manual_seed(seed)
    scheduler.set_timesteps(steps, device=device)

    tok = gemma_tokenizer(prompt, return_tensors="pt", padding="max_length",
                          truncation=True, max_length=MAX_GEMMA_LEN).to(device)
    ehs = gemma_model(**tok, output_hidden_states=True).hidden_states[-1].to(torch.float16)
    ehs = ehs * gemma_scale
    ehs_mask = tok.attention_mask.to(device)

    uncond = gemma_tokenizer("", return_tensors="pt", padding="max_length",
                            truncation=True, max_length=MAX_GEMMA_LEN).to(device)
    uncond_h = gemma_model(**uncond, output_hidden_states=True).hidden_states[-1].to(torch.float16)
    uncond_h = uncond_h * gemma_scale
    uncond_mask = uncond.attention_mask.to(device)

    ehs = torch.cat([uncond_h, ehs])
    ehs_mask = torch.cat([uncond_mask, ehs_mask])

    latents = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=torch.float16)
    latents = latents * scheduler.init_noise_sigma

    for t in tqdm(scheduler.timesteps, desc="Generating"):
        inp = torch.cat([latents] * 2)
        inp = scheduler.scale_model_input(inp, t)
        pred = unet(inp, t, encoder_hidden_states=ehs, encoder_attention_mask=ehs_mask).sample
        u, p = pred.chunk(2)
        pred = u + guidance * (p - u)
        latents = scheduler.step(pred, t, latents).prev_sample

    latents = latents / vae.config.scaling_factor
    img = vae.decode(latents).sample
    img = (img/2 + 0.5).clamp(0,1).cpu().permute(0,2,3,1).float().numpy()
    return Image.fromarray((img[0]*255).astype(np.uint8))

prompts = [
    "a cat sitting on a windowsill looking outside",
    "a watercolor painting of a mountain lake",
    "a neon-lit cyberpunk alleyway at night",
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p in zip(axes, prompts):
    print(f"Generating: {p}")
    img = generate(p, steps=30)
    ax.imshow(img)
    ax.set_title(p[:40]+"...", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{DRIVE_OUT}/samples.png", dpi=100)
plt.show()


In [ ]:
# @title 5.2 Save Research + Merged Inference Checkpoints
# Saves:
# 1) LoRA delta only (requires matching baked Gemma-UNet base)
# 2) Research checkpoint with ManualLoRA structure
# 3) Merged inference checkpoint with LoRA folded into base K/V layers

import os

# LoRA delta only — not standalone. Requires matching baked Gemma-UNet base.
lora_w = {
    n: p.detach().cpu().clone()
    for n, p in unet.named_parameters()
    if "lora_A" in n or "lora_B" in n
}
lora_delta_path = f"{DRIVE_OUT}/gemma3_sd_lora_delta_only.pt"
torch.save({
    "lora_state_dict": lora_w,
    "note": "LoRA delta only; requires matching baked Gemma-UNet base and ManualLoRA wrapping.",
    "gemma_model_id": gemma_path,
    "gemma_hidden_size": gemma_hidden_size,
    "max_gemma_len": MAX_GEMMA_LEN,
    "uses_kv_bias": True,
    "calibration_path": f"{DRIVE_OUT}/clip_gemma_calib.pt",
}, lora_delta_path)
print(f"LoRA delta-only checkpoint saved: {lora_delta_path}")

# Research checkpoint — reload requires ManualLoRA class + same architecture surgery.
research_path = f"{DRIVE_OUT}/gemma3_sd_research_checkpoint.pt"
torch.save({
    "unet_state_dict": {k: v.detach().cpu() for k, v in unet.state_dict().items()},
    "unet_config": dict(unet.config),
    "lora_state_dict": lora_w,
    "uses_manual_lora": True,
    "uses_kv_bias": True,
    "merged_lora": False,
    "gemma_model_id": gemma_path,
    "gemma_hidden_size": gemma_hidden_size,
    "new_cross_attention_dim": new_dim,
    "max_gemma_len": MAX_GEMMA_LEN,
    "gemma_scale": gemma_scale,
    "calibration_path": f"{DRIVE_OUT}/clip_gemma_calib.pt",
}, research_path)
print(f"Research checkpoint saved: {research_path}")

def merge_lora_linear(m):
    """Return base Linear with LoRA delta merged in-place."""
    if not isinstance(m, ManualLoRA):
        return m
    base = m.base
    delta = (m.lora_B.weight @ m.lora_A.weight) * m.scaling
    base.weight.data.add_(delta.to(device=base.weight.device, dtype=base.weight.dtype))
    return base

def merge_all_lora(module):
    """Recursively replace ManualLoRA children with merged base Linear layers."""
    for child_name, child in list(module.named_children()):
        if isinstance(child, ManualLoRA):
            setattr(module, child_name, merge_lora_linear(child))
        else:
            merge_all_lora(child)

merge_all_lora(unet)
print("LoRA merged into base to_k/to_v layers. ManualLoRA wrappers removed from live UNet.")

merged_path = f"{DRIVE_OUT}/gemma3_sd_inference_merged.pt"
torch.save({
    "unet_state_dict": {k: v.detach().cpu() for k, v in unet.state_dict().items()},
    "unet_config": dict(unet.config),
    "merged_lora": True,
    "uses_manual_lora": False,
    "uses_kv_bias": True,
    "gemma_model_id": gemma_path,
    "gemma_hidden_size": gemma_hidden_size,
    "new_cross_attention_dim": new_dim,
    "max_gemma_len": MAX_GEMMA_LEN,
    "gemma_scale": gemma_scale,
    "calibration_path": f"{DRIVE_OUT}/clip_gemma_calib.pt",
    "reload_note": "Rebuild SD1.5 UNet, replace attn2.to_k/to_v with Linear(640, inner_dim, bias=True), then load_state_dict(strict=True).",
}, merged_path)
size_gb = os.path.getsize(merged_path) / 1e9
print(f"Merged inference checkpoint saved: {merged_path} ({size_gb:.2f} GB)")


In [ ]:
# @title 5.3 Reload Merged Checkpoint Smoke Test (optional)
# This proves the saved merged artifact can be reloaded outside the live training graph.
# It reconstructs the custom Gemma-sized K/V architecture, including bias=True.

RUN_RELOAD_SMOKE_TEST = False  # Set True after 5.2 if you want to verify reload immediately.

if RUN_RELOAD_SMOKE_TEST:
    import torch.nn as nn
    from diffusers import UNet2DConditionModel

    merged_path = f"{DRIVE_OUT}/gemma3_sd_inference_merged.pt"
    ckpt = torch.load(merged_path, map_location="cpu")
    reload_dim = ckpt["new_cross_attention_dim"]
    reload_uses_kv_bias = ckpt.get("uses_kv_bias", True)

    reloaded_unet = UNet2DConditionModel.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        subfolder="unet",
        torch_dtype=unet_dtype,
    ).to(device)

    def apply_gemma_kv_shape_for_reload(model, new_dim, use_kv_bias=True):
        replacements = 0
        for name, module in model.named_modules():
            if hasattr(module, "attn2") and module.attn2 is not None:
                attn = module.attn2
                inner_dim = attn.to_k.out_features
                attn.to_k = nn.Linear(new_dim, inner_dim, bias=use_kv_bias, device=device, dtype=unet_dtype)
                attn.to_v = nn.Linear(new_dim, inner_dim, bias=use_kv_bias, device=device, dtype=unet_dtype)
                replacements += 1
        model.register_to_config(cross_attention_dim=new_dim)
        return replacements

    n = apply_gemma_kv_shape_for_reload(reloaded_unet, reload_dim, reload_uses_kv_bias)
    print(f"Reload surgery applied: {n} K/V pairs, dim={reload_dim}, bias={reload_uses_kv_bias}")
    reloaded_unet.load_state_dict(ckpt["unet_state_dict"], strict=True)
    reloaded_unet.eval()
    print("Merged checkpoint reload: PASS (strict=True)")
else:
    print("Reload smoke test skipped. Set RUN_RELOAD_SMOKE_TEST=True after saving to verify strict reload.")


## Roadmap & Next Steps

Current notebook is a proof-of-life custom Diffusers training loop, not a kohya training-loop port.

Immediate run target:
- Keep `MAX_SAMPLES = 2_000`, `FULLRANK_EPOCHS = 1`, `num_epochs = 1` for smoke test.
- Verify calibration train/val metrics, forward pass, loss logging, and merged checkpoint reload.

Scaling target:
- Increase calibration captions and dataset samples.
- Increase full-rank warmup epochs/steps.
- Consider better token alignment or pooled calibration if validation cosine is weak.
- For production training, either harden this custom loop or build a real sd-scripts Gemma strategy.


## Appendix: download_hf_dataset.py

Run this cell to write the download script to disk, then execute it above.

In [ ]:
# @title Appendix: Alternative Dataset (not used)
# This cell is a reference only. The training pipeline uses
# jackyhate/text-to-image-2M via streaming. No download needed.
# To switch datasets, change the load_dataset() call in cells 17/20.
